# Chapter 26: Deep Learning Fundamentals (Industry Standard)

## Learning Objectives\n- Build computational graph and autograd intuition\n- Implement MLP forward/backward from scratch\n- Stabilize training with initialization/regularization\n- Diagnose common training failures

## Prerequisites\n- Python basics and functions\n- Numpy basics for deep learning chapters\n- Understanding of earlier chapters (0-19)\n

# Chapter 26: Deep Learning Fundamentals (Not Surface-Level)

## Why this chapter matters
Deep learning success depends on correct optimization, initialization, normalization, and debugging practice, not just stacking layers.

## Learning goals
1. Build reverse-mode autodiff intuition.
2. Implement MLP training loop end-to-end.
3. Understand initialization, activations, normalization, and regularization.
4. Diagnose unstable training.

## Step 1: Computational graph mental model
Every forward operation creates a node.
Backward pass applies chain rule in reverse topological order.

## Step 2: Core training loop
1. forward pass
2. loss computation
3. backward pass
4. optimizer step
5. gradient reset

## Step 3: Stabilization stack
- initialization: Xavier/He
- normalization: BatchNorm/LayerNorm
- optimizer: AdamW baseline
- regularization: dropout + weight decay
- clipping: gradient norm clipping

## Step 4: Debugging checklist
1. overfit tiny batch test
2. inspect gradient norms by layer
3. verify label mapping
4. ensure train/eval mode handling
5. check leakage in validation pipeline

## Step 5: Practical milestones
1. scalar autograd engine
2. two-layer MLP classifier
3. multiclass cross-entropy implementation
4. mini-batch SGD/AdamW with learning-rate schedule

## Assignment
- Implement MLP on Iris from scratch (or numpy-accelerated loops).
- Log train loss, val loss, gradient norm.
- Add early stopping and model checkpointing.



## Part A: Scalar Autograd Engine

In [ ]:
"""Educational scalar autograd engine inspired by reverse-mode AD."""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Callable, List, Set


@dataclass(eq=False)
class Value:
    data: float
    grad: float = 0.0
    _prev: Set["Value"] = field(default_factory=set)
    _backward: Callable[[], None] = lambda: None

    def __hash__(self) -> int:
        return id(self)

    def __add__(self, other: "Value") -> "Value":
        out = Value(self.data + other.data, _prev={self, other})

        def _bw() -> None:
            self.grad += out.grad
            other.grad += out.grad

        out._backward = _bw
        return out

    def __mul__(self, other: "Value") -> "Value":
        out = Value(self.data * other.data, _prev={self, other})

        def _bw() -> None:
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _bw
        return out

    def relu(self) -> "Value":
        out = Value(self.data if self.data > 0 else 0.0, _prev={self})

        def _bw() -> None:
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad

        out._backward = _bw
        return out

    def backward(self) -> None:
        topo: List[Value] = []
        visited: Set[Value] = set()

        def build(v: Value) -> None:
            if v in visited:
                return
            visited.add(v)
            for p in v._prev:
                build(p)
            topo.append(v)

        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


def main() -> None:
    a = Value(2.0)
    b = Value(-3.0)
    c = Value(10.0)
    d = a * b + c
    e = d.relu()
    e.backward()

    print("e:", e.data)
    print("grad a:", a.grad)
    print("grad b:", b.grad)
    print("grad c:", c.grad)


if __name__ == "__main__":
    main()


## Part A Checkpoint\n- You can trace gradients through a small graph.\n- You can validate chain rule behavior numerically.

## Part B: MLP Classifier

In [ ]:
"""Numpy MLP classifier with manual forward/backward.
This is a bridge from pure Python math to modern DL training loops.
"""

from __future__ import annotations

import numpy as np


class MLP:
    def __init__(self, in_dim: int, hidden: int, out_dim: int, seed: int = 42) -> None:
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, np.sqrt(2 / in_dim), size=(in_dim, hidden))
        self.b1 = np.zeros((1, hidden))
        self.W2 = rng.normal(0, np.sqrt(2 / hidden), size=(hidden, out_dim))
        self.b2 = np.zeros((1, out_dim))

    def forward(self, X: np.ndarray):
        z1 = X @ self.W1 + self.b1
        a1 = np.maximum(0, z1)
        z2 = a1 @ self.W2 + self.b2
        ex = np.exp(z2 - np.max(z2, axis=1, keepdims=True))
        probs = ex / np.sum(ex, axis=1, keepdims=True)
        cache = (X, z1, a1, probs)
        return probs, cache

    @staticmethod
    def loss_and_grad(probs: np.ndarray, y: np.ndarray):
        n = len(y)
        loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))
        dlogits = probs.copy()
        dlogits[np.arange(n), y] -= 1
        dlogits /= n
        return loss, dlogits

    def backward(self, cache, dlogits):
        X, z1, a1, _ = cache
        dW2 = a1.T @ dlogits
        db2 = np.sum(dlogits, axis=0, keepdims=True)
        da1 = dlogits @ self.W2.T
        dz1 = da1 * (z1 > 0)
        dW1 = X.T @ dz1
        db1 = np.sum(dz1, axis=0, keepdims=True)
        return dW1, db1, dW2, db2

    def step(self, grads, lr: float, weight_decay: float = 0.0):
        dW1, db1, dW2, db2 = grads
        self.W1 -= lr * (dW1 + weight_decay * self.W1)
        self.b1 -= lr * db1
        self.W2 -= lr * (dW2 + weight_decay * self.W2)
        self.b2 -= lr * db2


def make_toy_data(n: int = 300, seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, 2))
    y = (X[:, 0] * X[:, 1] > 0).astype(int)
    return X, y


def accuracy(model: MLP, X: np.ndarray, y: np.ndarray) -> float:
    probs, _ = model.forward(X)
    pred = np.argmax(probs, axis=1)
    return float(np.mean(pred == y))


if __name__ == "__main__":
    X, y = make_toy_data(600)
    split = 500
    X_train, y_train = X[:split], y[:split]
    X_val, y_val = X[split:], y[split:]

    model = MLP(in_dim=2, hidden=32, out_dim=2)
    lr = 0.05

    for epoch in range(1, 401):
        probs, cache = model.forward(X_train)
        loss, dlogits = model.loss_and_grad(probs, y_train)
        grads = model.backward(cache, dlogits)
        model.step(grads, lr=lr, weight_decay=1e-4)

        if epoch % 40 == 0:
            train_acc = accuracy(model, X_train, y_train)
            val_acc = accuracy(model, X_val, y_val)
            print(f"epoch={epoch:03d} loss={loss:.4f} train_acc={train_acc:.3f} val_acc={val_acc:.3f}")


## Guided Exercise\nLog gradient norm per epoch and add simple early stopping based on validation loss plateau.

In [ ]:
# TODO (Student Exercise)
# Add gradient norm tracking inside training loop in the MLP script cell above.
pass

## Quick Quiz\n\n**Q1.** What does backward pass compute?\n\n**Answer:** Gradient of final loss with respect to each parameter.\n\n**Q2.** Why He initialization with ReLU?\n\n**Answer:** It keeps activation variance more stable across layers.\n\n**Q3.** Why overfit-a-tiny-batch test?\n\n**Answer:** It quickly validates correctness of forward/backward and optimization.\n

## Homework\nTrain on Iris (or toy data), add early stopping, and report train/val curves.